# Chapter 1 — Arrays and Strings

**Core ideas**
- **Hash tables** map keys to values with `O(1)` average lookup. In Python: `dict`, `set`, `collections.Counter`.
- **Dynamic arrays** (`list`) give `O(1)` amortized append (see Chapter 0).
- **String building**: Python strings are immutable, so `s += c` in a loop can be `O(N²)`. Collect pieces in a list and `"".join(...)` — the Python equivalent of Java's `StringBuilder`.

Below: a minimal hash table implementation, then problems 1.1–1.9.

In [1]:
from dataclasses import dataclass
from typing import Generic, Hashable, Iterator, TypeVar

K = TypeVar("K", bound=Hashable)
V = TypeVar("V")


class HashTable(Generic[K, V]):
    """Separate-chaining hash table with resizing at load factor 0.75."""

    def __init__(self, capacity: int = 8) -> None:
        self._buckets: list[list[tuple[K, V]]] = [[] for _ in range(capacity)]
        self._size = 0

    def _bucket(self, key: K) -> list[tuple[K, V]]:
        return self._buckets[hash(key) % len(self._buckets)]

    def __setitem__(self, key: K, value: V) -> None:
        bucket = self._bucket(key)
        for i, (k, _) in enumerate(bucket):
            if k == key:
                bucket[i] = (key, value)
                return
        bucket.append((key, value))
        self._size += 1
        if self._size > 0.75 * len(self._buckets):
            self._rehash()

    def __getitem__(self, key: K) -> V:
        for k, v in self._bucket(key):
            if k == key:
                return v
        raise KeyError(key)

    def __delitem__(self, key: K) -> None:
        bucket = self._bucket(key)
        for i, (k, _) in enumerate(bucket):
            if k == key:
                del bucket[i]
                self._size -= 1
                return
        raise KeyError(key)

    def __contains__(self, key: object) -> bool:
        try:
            self[key]  # type: ignore[index]
            return True
        except KeyError:
            return False

    def __len__(self) -> int:
        return self._size

    def __iter__(self) -> Iterator[K]:
        for bucket in self._buckets:
            for k, _ in bucket:
                yield k

    def _rehash(self) -> None:
        old = [pair for bucket in self._buckets for pair in bucket]
        self._buckets = [[] for _ in range(2 * len(self._buckets))]
        self._size = 0
        for k, v in old:
            self[k] = v


ht: HashTable[str, int] = HashTable()
for i, word in enumerate("the quick brown fox jumps over the lazy dog".split()):
    ht[word] = i
assert ht["the"] == 6 and len(ht) == 8 and "fox" in ht
del ht["fox"]
assert "fox" not in ht
print(sorted(ht))

['brown', 'dog', 'jumps', 'lazy', 'over', 'quick', 'the']


## 1.1 Is Unique
Decide whether every character in a string is distinct. Follow-up: what if you can't use extra data structures?

In [2]:
def is_unique(s: str) -> bool:
    """O(N) time, O(min(N, alphabet)) space."""
    return len(set(s)) == len(s)


def is_unique_early_exit(s: str) -> bool:
    seen: set[str] = set()
    for c in s:
        if c in seen:
            return False
        seen.add(c)
    return True


def is_unique_bitvector(s: str) -> bool:
    """Assumes lowercase a-z. Uses one int as a 26-bit set."""
    checker = 0
    for c in s:
        bit = 1 << (ord(c) - ord("a"))
        if checker & bit:
            return False
        checker |= bit
    return True


def is_unique_no_extra_space(s: str) -> bool:
    """Sort then compare neighbours: O(N log N). (Python's sorted() does allocate; in-place in C/Java.)"""
    chars = sorted(s)
    return all(a != b for a, b in zip(chars, chars[1:]))


for fn in (is_unique, is_unique_early_exit, is_unique_bitvector, is_unique_no_extra_space):
    assert fn("abcdef") and not fn("abcdea") and fn("")
print("1.1 ok")

1.1 ok


## 1.2 Check Permutation
Given two strings, decide whether one is a rearrangement of the other.

In [3]:
from collections import Counter


def check_permutation(a: str, b: str) -> bool:
    return len(a) == len(b) and Counter(a) == Counter(b)


def check_permutation_sort(a: str, b: str) -> bool:
    return sorted(a) == sorted(b)


for fn in (check_permutation, check_permutation_sort):
    assert fn("dog", "god") and not fn("dog", "dogs") and not fn("aab", "abb")
print("1.2 ok")

1.2 ok


## 1.3 URLify
Replace every space with `%20`. The book's version works in-place on a character array that has enough trailing room, using the "true length"; we show that approach on a `list[str]` plus the idiomatic one-liner.

In [4]:
def urlify(chars: list[str], true_length: int) -> list[str]:
    """In-place, filling from the back. O(N)."""
    spaces = chars[:true_length].count(" ")
    index = true_length + 2 * spaces
    for i in range(true_length - 1, -1, -1):
        if chars[i] == " ":
            chars[index - 3:index] = list("%20")
            index -= 3
        else:
            chars[index - 1] = chars[i]
            index -= 1
    return chars


def urlify_pythonic(s: str) -> str:
    return s.strip().replace(" ", "%20")


buf = list("Mr John Smith    ")
assert "".join(urlify(buf, 13)) == "Mr%20John%20Smith"
assert urlify_pythonic("Mr John Smith    ") == "Mr%20John%20Smith"
print("1.3 ok")

1.3 ok


## 1.4 Palindrome Permutation
Could the letters of a phrase be rearranged into a palindrome? Ignore case and non-letters. A palindrome allows at most one character with an odd count.

In [5]:
def palindrome_permutation(phrase: str) -> bool:
    counts = Counter(c for c in phrase.lower() if c.isalpha())
    return sum(v % 2 for v in counts.values()) <= 1


def palindrome_permutation_bits(phrase: str) -> bool:
    """Toggle one bit per letter; at most one bit may remain set."""
    bits = 0
    for c in phrase.lower():
        if "a" <= c <= "z":
            bits ^= 1 << (ord(c) - ord("a"))
    return bits & (bits - 1) == 0


for fn in (palindrome_permutation, palindrome_permutation_bits):
    assert fn("Tact Coa") and fn("") and not fn("abc")
print("1.4 ok")

1.4 ok


## 1.5 One Away
Three edits exist: insert, remove, or replace one character. Are two strings at most one edit apart?

In [6]:
def one_away(a: str, b: str) -> bool:
    if abs(len(a) - len(b)) > 1:
        return False
    if len(a) > len(b):
        a, b = b, a  # a is the shorter
    i = j = 0
    found_difference = False
    while i < len(a) and j < len(b):
        if a[i] != b[j]:
            if found_difference:
                return False
            found_difference = True
            if len(a) == len(b):
                i += 1  # replace: advance both
        else:
            i += 1
        j += 1  # always advance the longer string
    return True


assert one_away("pale", "ple")
assert one_away("pales", "pale")
assert one_away("pale", "bale")
assert not one_away("pale", "bake")
assert one_away("", "a") and not one_away("", "ab")
print("1.5 ok")

1.5 ok


## 1.6 String Compression
Compress runs of repeated characters: `aabcccccaaa` → `a2b1c5a3`. If the result isn't shorter, return the original.

In [7]:
from itertools import groupby


def compress(s: str) -> str:
    parts: list[str] = []  # the "StringBuilder"
    for char, run in groupby(s):
        parts.append(f"{char}{sum(1 for _ in run)}")
    compressed = "".join(parts)
    return compressed if len(compressed) < len(s) else s


assert compress("aabcccccaaa") == "a2b1c5a3"
assert compress("abc") == "abc"
assert compress("") == ""
print("1.6 ok")

1.6 ok


## 1.7 Rotate Matrix
Rotate an N×N matrix 90° clockwise, ideally in place.

In [8]:
def rotate_matrix(m: list[list[int]]) -> list[list[int]]:
    """In place, layer by layer, 4-way swap. O(N^2) time, O(1) extra space."""
    n = len(m)
    for layer in range(n // 2):
        first, last = layer, n - 1 - layer
        for i in range(first, last):
            offset = i - first
            top = m[first][i]
            m[first][i] = m[last - offset][first]        # left -> top
            m[last - offset][first] = m[last][last - offset]  # bottom -> left
            m[last][last - offset] = m[i][last]          # right -> bottom
            m[i][last] = top                             # top -> right
    return m


def rotate_matrix_transpose(m: list[list[int]]) -> list[list[int]]:
    """Transpose then reverse each row — also in place."""
    n = len(m)
    for i in range(n):
        for j in range(i + 1, n):
            m[i][j], m[j][i] = m[j][i], m[i][j]
    for row in m:
        row.reverse()
    return m


def rotate_pythonic(m: list[list[int]]) -> list[list[int]]:
    return [list(row) for row in zip(*m[::-1])]


grid = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
expected = [[7, 4, 1], [8, 5, 2], [9, 6, 3]]
assert rotate_pythonic(grid) == expected
assert rotate_matrix([r[:] for r in grid]) == expected
assert rotate_matrix_transpose([r[:] for r in grid]) == expected
big = [[r * 4 + c for c in range(4)] for r in range(4)]
assert rotate_matrix([r[:] for r in big]) == rotate_pythonic(big)
print("1.7 ok")

1.7 ok


## 1.8 Zero Matrix
If a cell of an M×N matrix is 0, zero out its whole row and column. Pitfall: don't let newly-written zeros cascade. The O(1)-space trick uses the first row/column as flag storage.

In [9]:
def zero_matrix(m: list[list[int]]) -> list[list[int]]:
    if not m:
        return m
    rows, cols = len(m), len(m[0])
    first_row_zero = any(m[0][c] == 0 for c in range(cols))
    first_col_zero = any(m[r][0] == 0 for r in range(rows))
    for r in range(1, rows):
        for c in range(1, cols):
            if m[r][c] == 0:
                m[r][0] = m[0][c] = 0
    for r in range(1, rows):
        if m[r][0] == 0:
            m[r] = [0] * cols
    for c in range(1, cols):
        if m[0][c] == 0:
            for r in range(rows):
                m[r][c] = 0
    if first_row_zero:
        m[0] = [0] * cols
    if first_col_zero:
        for r in range(rows):
            m[r][0] = 0
    return m


assert zero_matrix([[1, 2, 3], [4, 0, 6], [7, 8, 9]]) == [[1, 0, 3], [0, 0, 0], [7, 0, 9]]
assert zero_matrix([[0, 2], [3, 4]]) == [[0, 0], [0, 4]]
print("1.8 ok")

1.8 ok


## 1.9 String Rotation
Given an `is_substring` helper, check whether `s2` is a rotation of `s1` using only one call to it. Key insight: every rotation of `s1` appears inside `s1 + s1`.

In [10]:
def is_substring(big: str, small: str) -> bool:
    return small in big


def is_rotation(s1: str, s2: str) -> bool:
    return len(s1) == len(s2) and is_substring(s1 + s1, s2)


assert is_rotation("waterbottle", "erbottlewat")
assert not is_rotation("waterbottle", "erbottlewta")
print("1.9 ok")

1.9 ok
